# АэроРоуд — обучение YOLO11n в Google Colab

1. Среда выполнения → Сменить среду → **T4 GPU**.
2. Запустите ячейки по порядку. В первой загрузите `hazards_dataset.zip` (папка `data/hazards`, которую сделал `prepare_dataset.py`).
3. В конце скачается `best.pt` — положите его в `backend/models/best.pt` и перезапустите сервер.

In [ ]:
!pip -q install -U "ultralytics>=8.3.0"
from google.colab import files
up = files.upload()  # выберите hazards_dataset.zip
name = next(iter(up))
!rm -rf /content/hazards && mkdir -p /content/hazards && unzip -q -o "$name" -d /content/hazards
!find /content/hazards -maxdepth 2 -name hazards.yaml

In [ ]:
import glob, os
yaml_path = glob.glob('/content/hazards/**/hazards.yaml', recursive=True)[0]
root = os.path.dirname(yaml_path)
text = open(yaml_path, encoding='utf-8').read()
open('/content/data.yaml', 'w', encoding='utf-8').write(f'path: {root}\n' + text)
print(open('/content/data.yaml').read())
print(open(os.path.join(root, 'REPORT.md'), encoding='utf-8').read())

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')  # или yolo11s.pt — точнее, но медленнее
model.train(
    data='/content/data.yaml', epochs=150, patience=40, imgsz=640, batch=16,
    degrees=10, flipud=0.0, fliplr=0.5, mosaic=1.0, close_mosaic=10, hsv_v=0.4, scale=0.5,
    project='/content/runs', name='hazards', exist_ok=True, seed=0,
)
# Если в датасете только съёмка сверху: degrees=180, flipud=0.5

In [ ]:
best = YOLO(model.trainer.best)
m = best.val(data='/content/data.yaml', split='val', conf=0.001, iou=0.6)
print('| Класс | P | R | mAP@0.5 | mAP@0.5:0.95 |')
for i, c in enumerate(m.box.ap_class_index):
    p, r, a50, a = m.box.class_result(i)
    print(f'| {best.names[int(c)]} | {p:.3f} | {r:.3f} | {a50:.3f} | {a:.3f} |')
print(f'| все | {m.box.mp:.3f} | {m.box.mr:.3f} | {m.box.map50:.3f} | {m.box.map:.3f} |')

In [ ]:
from IPython.display import Image, display
for f in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    p = f'/content/runs/hazards/{f}'
    if os.path.exists(p): display(Image(p, width=900))

In [ ]:
files.download(model.trainer.best)  # → backend/models/best.pt